# Statistical Decision Theory: From Beliefs to Actions

Companion notebook for Tutorial 3, Chapter 20. Runs every code block from the chapter; read the chapter for the full explanations.

Chapter: https://josephausterweil.github.io/probintro/intro2/20_statistical_decision_theory/

In [ ]:
# Colab setup
!pip -q install "genjax==0.10.3" "jax==0.5.3" 2>/dev/null

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr

# states theta in {fresh(0), stale(1)}; actions {eat(0), toss(1)}
L = jnp.array([[0.0, 3.0],      # theta = fresh:  eat -> 0,   toss -> 3
               [10.0, 1.0]])     # theta = stale:  eat -> 10,  toss -> 1
belief = jnp.array([0.9, 0.1])   # belief over theta: P(fresh) = 0.9, P(stale) = 0.1
acts = ["eat", "toss"]

In [ ]:
exp_loss   = belief @ L                 # belief-weighted (expected) loss: sum over theta of P(theta)*L[theta, a]
worst_case = jnp.max(L, axis=0)         # max over theta (axis 0 = the rows): one worst case per action

print("             eat   toss")
print(f"E[L]       {float(exp_loss[0]):5.2f} {float(exp_loss[1]):5.2f}   ->  Bayes picks {acts[int(jnp.argmin(exp_loss))]}")
print(f"max L      {float(worst_case[0]):5.2f} {float(worst_case[1]):5.2f}   ->  minimax picks {acts[int(jnp.argmin(worst_case))]}")

In [ ]:
grid = jnp.linspace(0.0, 10.0, 1001)             # candidate weights / estimates (×100 g)
dens = grid**2.3 * jnp.exp(-grid / 1.15)         # a skewed posterior over the weight
dens = dens / jnp.trapezoid(dens, grid)          # normalize it

# the three summaries, read straight off the posterior
mode   = grid[jnp.argmax(dens)]
mean   = jnp.trapezoid(grid * dens, grid)
cdf    = jnp.cumsum(dens) * (grid[1] - grid[0])
median = grid[jnp.argmin(jnp.abs(cdf - 0.5))]
print(f"read off the posterior:   mode = {float(mode):.2f}   mean = {float(mean):.2f}   median = {float(median):.2f}")

# now DERIVE each as the Bayes estimator: the action a minimizing expected loss
def bayes_estimator(loss):
    T, A = grid[None, :], grid[:, None]                       # theta (cols) vs action (rows)
    expected_loss = jnp.trapezoid(loss(T, A) * dens[None, :], grid, axis=1)
    return grid[jnp.argmin(expected_loss)]

a_01  = bayes_estimator(lambda t, a: (jnp.abs(t - a) > 0.05).astype(float))   # 0–1 loss
a_sq  = bayes_estimator(lambda t, a: (t - a) ** 2)                            # squared loss
a_abs = bayes_estimator(lambda t, a: jnp.abs(t - a))                          # absolute loss
print(f"argmin 0–1 loss      = {float(a_01):.2f}   (lands on the mode)")
print(f"argmin squared loss  = {float(a_sq):.2f}   (lands on the mean)")
print(f"argmin absolute loss = {float(a_abs):.2f}   (lands on the median)")

In [ ]:
from genjax import gen, beta as gbeta

@gen
def freshness_posterior():
    p_stale = gbeta(2.0, 12.0) @ "p_stale"   # belief about P(stale): mean 2/14 ≈ 0.14
    return p_stale

def expected_loss_by_sampling(key, n=20000):
    keys = jr.split(key, n)
    ps = jax.vmap(lambda k: freshness_posterior.simulate(k, ()).get_retval())(keys)
    el_eat  = jnp.mean((1 - ps) * L[0, 0] + ps * L[1, 0])   # average loss of eating
    el_toss = jnp.mean((1 - ps) * L[0, 1] + ps * L[1, 1])   # average loss of tossing
    return el_eat, el_toss

el_eat, el_toss = expected_loss_by_sampling(jr.key(0))
print(f"Monte-Carlo expected loss:   eat = {float(el_eat):.2f}   toss = {float(el_toss):.2f}")
print(f"Bayes action by sampling: {acts[int(jnp.argmin(jnp.array([el_eat, el_toss])))]}")

In [ ]:
import jax.numpy as jnp
from jax.scipy.special import gammaln

def p_correct(k, p):
    # P(majority of k samples from your belief favors the better option), ties split fairly.
    # p = your belief that the better option really is better.
    j = jnp.arange(k + 1)
    log_choose = gammaln(k + 1.) - gammaln(j + 1.) - gammaln(k - j + 1.)
    pmf = jnp.exp(log_choose + j * jnp.log(p) + (k - j) * jnp.log1p(-p))
    win = (j > k / 2) + 0.5 * (j == k / 2)
    return float(jnp.sum(pmf * win))

In [ ]:
p, cost = 0.75, 0.1               # belief: 75% sure; each sample costs 0.1 of a decision's time
ks   = list(range(1, 13))
acc  = [p_correct(k, p) for k in ks]
rate = [a / (1 + cost * k) for a, k in zip(acc, ks)]
best = ks[int(jnp.argmax(jnp.array(rate)))]

print(" k   P(correct)   reward rate")
for k, a, r in zip(ks, acc, rate):
    print(f"{k:2d}     {a:.3f}        {r:.3f}{'   <- best' if k == best else ''}")
print(f"\noptimal number of samples: k* = {best}  (one and done)")

# what the k=1 policy actually DOES: it follows its single sample, so it picks the
# option it believes is better with probability p -- it MATCHES its belief.
print(f"one-and-done picks the believed-better option with prob {p:.2f}  (probability matching)")
print(f"'always pick the more likely' (maximizing) would pick it with prob 1.00")